# 03 — Slope Distribution Plots

Visualise the distribution of practice-effect slopes across parcels.
Three perspectives:
1. **Within-subject** — histogram + volcano plot per subject × task/contrast
2. **Pooled** — all subjects' slopes combined
3. **Group-averaged** — volcano plot of group mean slopes, colored by proportion of significant subjects

In [ ]:
import sys
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import TASKS, CONTRASTS, SUBJECTS, RT_CONTRAST, DATA_DIR, FIGURES_DIR

with open(DATA_DIR / 'surface_indiv_slopes.pkl', 'rb') as f:
    indiv_slopes = pickle.load(f)
with open(DATA_DIR / 'surface_avg_slopes.pkl', 'rb') as f:
    avg_slopes = pickle.load(f)

# Parcel list from first available entry
for subj in SUBJECTS:
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast != RT_CONTRAST and indiv_slopes.get(subj, {}).get(task, {}).get(contrast):
                parcel_names = list(indiv_slopes[subj][task][contrast].keys())
                break
        else:
            continue
        break

FIG_DIR = FIGURES_DIR / 'slope_distributions'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Parcels: {len(parcel_names)}')

In [ ]:
def plot_within_subject_effects(task, contrast, subject):
    data = indiv_slopes.get(subject, {}).get(task, {}).get(contrast, {})
    if not data:
        return

    slopes  = np.array([data[p]['beta_slope'] for p in parcel_names if p in data])
    pvals   = np.array([data[p]['p_value']    for p in parcel_names if p in data])
    sig     = pvals < 0.05

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{subject} | {task} | {contrast}', fontsize=11)

    # Histogram
    ax = axes[0]
    ax.hist(slopes[~sig], bins=30, color='steelblue', alpha=0.7, label='n.s.')
    ax.hist(slopes[sig],  bins=30, color='firebrick', alpha=0.7, label='p<0.05')
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Slope (β/encounter)'); ax.set_ylabel('Parcels')
    ax.set_title(f'n={len(slopes)}  sig={sig.sum()}  mean={slopes.mean():.3f}')
    ax.legend(fontsize=8)

    # Volcano
    ax = axes[1]
    ax.scatter(slopes[~sig], -np.log10(pvals[~sig]), c='steelblue', s=8, alpha=0.5)
    ax.scatter(slopes[sig],  -np.log10(pvals[sig]),  c='firebrick', s=8, alpha=0.8)
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.axhline(-np.log10(0.05), color='gray', lw=1, ls=':')
    ax.set_xlabel('Slope'); ax.set_ylabel('-log10(p)')
    ax.set_title('Volcano plot')

    plt.tight_layout()
    out = FIG_DIR / 'within_subject' / f'{subject}_{task}_{contrast}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=100, bbox_inches='tight')
    plt.close()

In [ ]:
def plot_group_averaged_effects(task, contrast):
    data = avg_slopes.get(task, {}).get(contrast, {})
    if not data:
        return

    parcels_present = [p for p in parcel_names if p in data]
    slopes   = np.array([data[p]['slope_mean']                          for p in parcels_present])
    pvals    = np.array([data[p]['group_p_value']                       for p in parcels_present])
    prop_sig = np.array([data[p]['individual_p_significant_proportion'] for p in parcels_present])
    sig      = pvals < 0.05

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Group | {task} | {contrast}', fontsize=11)

    # Histogram
    ax = axes[0]
    ax.hist(slopes[~sig], bins=30, color='steelblue', alpha=0.7, label='n.s.')
    ax.hist(slopes[sig],  bins=30, color='firebrick', alpha=0.7, label='p<0.05')
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('Mean slope'); ax.set_ylabel('Parcels')
    ax.set_title(f'n={len(slopes)}  group sig={sig.sum()}')
    ax.legend(fontsize=8)

    # Volcano colored by proportion of significant subjects
    ax = axes[1]
    sc = ax.scatter(slopes, -np.log10(pvals), c=prop_sig,
                    cmap='YlOrRd', s=10, vmin=0, vmax=1, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Prop. subjects p<0.05')
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.axhline(-np.log10(0.05), color='gray', lw=1, ls=':')
    ax.set_xlabel('Group mean slope'); ax.set_ylabel('-log10(p)')
    ax.set_title('Group volcano')

    plt.tight_layout()
    out = FIG_DIR / 'group_averaged' / f'{task}_{contrast}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=100, bbox_inches='tight')
    plt.close()

In [ ]:
for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for subject in SUBJECTS:
            plot_within_subject_effects(task, contrast, subject)
        plot_group_averaged_effects(task, contrast)
    print(f'  Done: {task}')

print(f'Figures saved to {FIG_DIR}')